# Train the intent adapter — AdapterOps Phase 0

QLoRA over Qwen2.5-1.5B-Instruct on Banking77 intent classification.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Free tier is enough.

Trains from the *frozen* splits committed in the repo, so it cannot see a golden-set row.
Also saves an **under-trained checkpoint at 15% of steps** — the subtle regression the
Phase 5 gate-sensitivity test (M11) needs. Free to keep now, a full retrain later.

In [ ]:
# 1. Confirm we have a GPU. Everything below needs CUDA.
import subprocess

out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=False).stdout.strip()
print(out or "NO GPU - set Runtime > Change runtime type > T4 GPU, then rerun")

In [ ]:
# 2. Clone the repo. The frozen splits are committed, so there is no separate download.
!git clone --depth 1 https://github.com/tpawar03/AdapterOps.git
%cd AdapterOps
!ls -la data/intent/

In [ ]:
# 3. Install deps. Colab already has torch built for its CUDA - don't let pip replace it.
!pip install -q "transformers>=4.44" "peft>=0.13" "datasets>=2.21" "accelerate>=0.34" \
               "bitsandbytes>=0.44" pandas pyarrow

# Put the package on sys.path directly rather than `pip install -e .`. An editable install
# drops a .pth file into site-packages, and .pth files are only read when the interpreter
# starts - a kernel that is already running never sees it. This needs no restart.
import os
import sys

SRC = os.path.abspath("src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print("on path:", SRC)

In [ ]:
# 4. Sanity-check the data before spending GPU time on it.
import adapterops
from adapterops.train.qlora import format_examples, load_split

print("adapterops loaded from:", adapterops.__file__)

train_df, val_df = load_split("intent", "train"), load_split("intent", "val")
print(f"train {len(train_df):,}   val {len(val_df):,}   classes {train_df.label_text.nunique()}")
print()
example = format_examples(train_df.head(1), "intent")[0]
print(example["prompt"])
print("->", example["completion"])

### Guard against the mistake that ruins everything downstream

If a golden-set row reaches training, every score afterwards is meaningless and nothing
will tell you. Check it explicitly rather than trusting the split script.

In [ ]:
# 5. The golden set must be disjoint from training data.
import pandas as pd

golden = pd.read_parquet("evals/golden/intent.parquet")
overlap = set(golden.text) & set(train_df.text)
assert not overlap, f"LEAK: {len(overlap)} golden rows found in train"
print(f"golden {len(golden)} rows, {golden.label_text.nunique()} classes, 0 overlap  OK")

In [ ]:
# 6. Train. Roughly 20-40 min on a free T4: 798 steps over 8,495 rows, 3 epochs.
from adapterops.train.qlora import TrainConfig, train

cfg = TrainConfig(task="intent", output_dir="checkpoints/intent")
summary = train(cfg)
summary

### Push to the Hub

The adapter becomes a versioned revision — the registry the system manifest will pin (F16).
Set your token first: Colab left sidebar → 🔑 Secrets → add `HF_TOKEN` with **Write** access.

In [ ]:
# 7. Authenticate and push both the trained adapter and the M11 under-trained one.
from google.colab import userdata
from huggingface_hub import HfApi, login

login(userdata.get("HF_TOKEN"))
api = HfApi()
user = api.whoami()["name"]

for path, repo in [
    ("checkpoints/intent", f"{user}/adapterops-intent"),
    ("checkpoints/intent-undertrained-m11", f"{user}/adapterops-intent-undertrained-m11"),
]:
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(folder_path=path, repo_id=repo)
    print("pushed", repo)

In [ ]:
# 8. Print the summary so it can be committed to runs/ back in the repo.
import json

print(json.dumps(summary, indent=2))

### Score it

Both numbers come from the same 770 frozen golden rows with identical decoding, so they
are directly comparable. The adapter is scored first, then the prompted baseline — the
base model with no adapter, all 77 labels listed and ten worked examples. **That baseline,
not the majority floor, is what decides whether fine-tuning earned its place (M2).**

In [ ]:
# 9. Score the adapter on the frozen golden set.
import json

import pandas as pd
import torch
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer

from adapterops.train.qlora import PROMPT

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", padding_side="left")
tok.pad_token = tok.pad_token or tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, "checkpoints/intent").eval()

golden = pd.read_parquet("evals/golden/intent.parquet")
labels = sorted(pd.read_parquet("data/intent/split_train.parquet").label_text.unique())

def score(make_prompt, batch, tag, mdl):
    preds = []
    for i in range(0, len(golden), batch):
        chunk = golden.iloc[i:i + batch]
        enc = tok([make_prompt(t) for t in chunk.text], return_tensors="pt",
                  padding=True).to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(**enc, max_new_tokens=12, do_sample=False,
                               pad_token_id=tok.pad_token_id)
        preds += [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)
                  .strip().split("\n")[0].strip() for o in out]
        print(f"\r{tag} {i + len(chunk)}/{len(golden)}", end="")
    gold = golden.label_text.tolist()
    res = {"system": tag, "n": len(gold),
           "micro_accuracy": accuracy_score(gold, preds),
           "macro_f1": f1_score(gold, preds, average="macro", zero_division=0),
           "exact_label_rate": sum(x in set(labels) for x in preds) / len(preds)}
    print("\n" + json.dumps(res, indent=2))
    return res

adapter_res = score(lambda t: PROMPT.format(text=t), 32, "adapter", model)

In [ ]:
# 10. Prompted baseline (M2): same base model, NO adapter, strong few-shot prompt.
SEED = 20260909
train_df = pd.read_parquet("data/intent/split_train.parquet")
demo = train_df.groupby("label_text", group_keys=False).head(1).sample(n=10, random_state=SEED)
assert not (set(demo.text) & set(golden.text)), "LEAK: demo drawn from golden set"
demos = "\n".join(f"Request: {r.text}\nIntent: {r.label_text}" for r in demo.itertuples())
HEADER = ("Classify the customer's banking request into exactly one intent label.\n"
          "Valid labels:\n" + ", ".join(labels) + "\n\nExamples:\n" + demos + "\n\n")

# batch 8, not 32: these prompts are ~772 tokens instead of ~34
baseline_res = score(lambda t: HEADER + f"Request: {t}\nIntent:", 8,
                     "prompted-baseline", model.get_base_model())

delta = adapter_res["micro_accuracy"] - baseline_res["micro_accuracy"]
print(f"\nadapter  {adapter_res['micro_accuracy']:.4f}")
print(f"baseline {baseline_res['micro_accuracy']:.4f}")
print(f"delta    {delta:+.4f}   <- this is the M2 number")

import pathlib

pathlib.Path("runs").mkdir(exist_ok=True)
for r in (adapter_res, baseline_res):
    pathlib.Path(f"runs/intent__{r['system']}.json").write_text(json.dumps(r, indent=2) + "\n")
print("\nwrote runs/intent__adapter.json and runs/intent__prompted-baseline.json")

### Next

1. Commit `runs/intent__adapter.json` and `runs/intent__prompted-baseline.json` back to the repo.
2. **Gate 0.5** — rent an A10G, serve this adapter under vLLM, then load a second adapter
   alongside it. If concurrent multi-LoRA fails there, the headline claim changes.
3. Latency and cost are measured on that rented box, never from this free tier.